# Building a **held-out validation** dataset from Rico et al. 2026 — `Experiment` objects ready to test the calibration

*Tutorial 2b in the series (follows `metagenomics.ipynb`, complements `build_experimental_db.ipynb`).*

The Ding et al. study (`build_experimental_db.ipynb`) was used to **calibrate** e-ADM. To know
whether that calibration *generalises*, we need an **independent** dataset the model never saw.

This notebook builds one from:

> Rico, Schmidt, Ghadermazi, Otto, Metcalf, Castillo Jaimes, Chan, Reardon, De Long.
> *Higher-order interaction effects among operating conditions and feedstocks shape reactor
> microbiomes and fatty acid production profiles.* **Bioresource Technology 457 (2026) 134980.**
> [doi:10.1016/j.biortech.2026.134980](https://doi.org/10.1016/j.biortech.2026.134980) — open access.

Everything we need is **already provided** by the paper's public repository
([github.com/jorgericko/interactions](https://github.com/jorgericko/interactions)) — denoised
16S rep-seqs, a feature table, fatty-acid time-series, and sample metadata — plus the raw reads
on SRA (`PRJNA1443009`). **We do not re-run the amplicon pipeline**: the rep-seqs and feature
table are the pipeline's *output*, so we enter downstream of DADA2 and only map ASVs → genomes →
functional-group COD.

### What makes this a good — but *different* — validation
Rico et al. is **arrested anaerobic digestion (AAD)**: methanogenesis is suppressed *by pH alone*
(inhibited at pH 5 and 9, active at pH 7) to redirect carbon toward fatty acids. So this dataset
tests the **hydrolysis → acidogenesis → chain-elongation + pH-inhibition** machinery of e-ADM —
*not* the methane pathway that Ding validated. That is exactly the point of a complementary
held-out test. Methane was measured only at the final timepoint, so we do **not** use it as a
fit target; the **VFA speciation trajectories (C2–C6)** are the observables.


## 1. Setup & the slice we validate on

The full study is a 3×2×2×2 factorial (pH × temperature × feedstock × inoculum) = 24 treatments ×
3 reps = 72 fed reactors — far too broad for a clean held-out test. We take the **cleanest slice**
that matches the training feedstock/inoculum family and isolates a single question:

> *Does e-ADM reproduce how bulk pH reshapes the VFA spectrum on food waste?*

- **feedstock = food waste** (as in Ding)
- **inoculum = A** (anaerobic digester / municipal wastewater sludge — closest to Ding's sludge)
- **temperature = 35 °C** (mesophilic, as in Ding)
- **pH ∈ {5, 7, 9}**, reps {A, B, C}  → **3 conditions × 3 replicates = 9 experiments**

pH is a *controlled* factor here, so it maps directly onto e-ADM's `S_H_ion` control state.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from adtoolbox import core, configs, utils

REPO = Path.cwd().parent if Path.cwd().name == "Examples" else Path.cwd()

RAW          = REPO / "Examples" / "Studies" / "rico_raw"     # provided github files (ship with repo)
DATABASE_DIR = REPO / "database"                              # ADToolbox databases (GTDB, SEED, ...)
SEED_OUT     = REPO / "tutorial_output" / "rico_day0"         # per-D0-sample cod_profile.csv (metagenomics seed)
MODEL_DB     = REPO / "reference_data" / "models.json"
DOI = "https://doi.org/10.1016/j.biortech.2026.134980"
SRA = "PRJNA1443009"

# --- the validation slice ---
FEEDSTOCK, INOCULUM, TEMPERATURE = "F", "A", 35
PHS  = [5, 7, 9]
REPS = ["A", "B", "C"]
VARIABLES = ["S_ac", "S_pro", "S_bu", "S_va", "S_cap"]        # C2, C3, C4, C5, C6 as model species

print("raw data dir:", RAW)
print("slice: food waste, sludge inoculum (A), 35 C, pH", PHS, "x reps", REPS)

## 2. What the paper reports (and the units)

The fatty-acid table `data_acids_production.csv` gives concentrations in **g L⁻¹** for
C2 (acetic), C3 (propionic), n-C4/iso-C4 (butyric), n-C5/iso-C5 (valeric), n-C6/iso-C6 (caproic),
and C7 (heptanoic), tagged by `inoc / T / feedstock / pH / rep` and a continuous sampling `day`.

Mapping onto e-ADM species (which lumps n- and iso- isomers into one pool):

| paper | + isomer | e-ADM species |
|-------|----------|---------------|
| C2    | —        | `S_ac`  (acetate)   |
| C3    | —        | `S_pro` (propionate)|
| C4    | ISOC4    | `S_bu`  (butyrate)  |
| C5    | ISOC5    | `S_va`  (valerate)  |
| C6    | ISOC6    | `S_cap` (caproate)  |
| C7    | —        | **dropped** — e-ADM has no heptanoate state |


In [ ]:
fa = pd.read_csv(RAW / "data_acids_production.csv")
sl = fa[(fa.feedstock == FEEDSTOCK) & (fa.inoc == INOCULUM) &
        (fa["T"] == TEMPERATURE) & (fa.pH.isin(PHS))].copy()

# lump n- + iso- isomers into the model pools; C7 has no model state and is dropped
sl["S_ac"]  = sl.C2
sl["S_pro"] = sl.C3
sl["S_bu"]  = sl.C4 + sl.ISOC4
sl["S_va"]  = sl.C5 + sl.ISOC5
sl["S_cap"] = sl.C6 + sl.ISOC6

print("slice rows:", len(sl))
print(sl.groupby(["pH", "rep"]).size().rename("n_timepoints").to_string())
print("\nsampling days seen:", sorted(sl.day.unique()))

## 3. COD — the model's currency, computed with ADToolbox's SEED database

e-ADM tracks every species as **gCOD L⁻¹**. We convert each VFA from g L⁻¹ using the *same*
SEED-database COD factors the Ding notebook used, so the two datasets are on one identical scale.
`cod_calc` returns gCOD per mol-equivalent; `/1000` gives the mg L⁻¹ → gCOD L⁻¹ factor, and since
the paper reports **g L⁻¹** we multiply by 1000 more (`× COD_FACTOR × 1000`, i.e. gCOD per g).


In [ ]:
seed_db = core.SeedDB(configs.Database(database_dir=str(DATABASE_DIR)))

SEED_ID = {"S_ac": "cpd00029", "S_pro": "cpd00141", "S_bu": "cpd00211",
           "S_va": "cpd00597", "S_cap": "cpd01113"}                 # acetate ... caproate

COD_FACTOR = {sp: seed_db.instantiate_metabs(cid).cod_calc(add_h=1) / 1000
              for sp, cid in SEED_ID.items()}                       # gCOD per mg
GCOD_PER_G = {sp: COD_FACTOR[sp] * 1000.0 for sp in VARIABLES}      # gCOD per g (paper units)

print("VFA COD factors from the SEED database:")
for sp in VARIABLES:
    print(f"  {sp:6} {SEED_ID[sp]}   {COD_FACTOR[sp]:.6f} gCOD/mg   ({GCOD_PER_G[sp]:.3f} gCOD/g)")

## 4. VFA trajectories per (pH, rep), in gCOD L⁻¹

Each replicate reactor becomes one time-series. A few reactors have two measurements at the same
nominal day (e.g. 11.29 vs 11.3) — we average those so `time` is strictly increasing, which the
integrator's `t_eval` requires.


In [ ]:
def trajectory(ph, rep):
    """Return (times, data[time, VFA]) in gCOD/L for one reactor."""
    r = sl[(sl.pH == ph) & (sl.rep == rep)].copy()
    r["day"] = r["day"].round(1)                                    # collapse 11.29/11.3 -> 11.3
    g = r.groupby("day")[VARIABLES].mean().sort_index()            # average duplicate days
    gcod = g.to_numpy(float) * np.array([GCOD_PER_G[v] for v in VARIABLES])
    return g.index.to_numpy(float), gcod

t, d = trajectory(5, "A")
print("pH 5, rep A  —  days:", list(t))
print(pd.DataFrame(d, index=t, columns=VARIABLES).round(3).to_string())
print("\nAt pH 5 butyrate (S_bu) + caproate (S_cap) dominate — the chain-elongation signature "
      "the paper reports for food waste at 35 C.")

## 5. pH as a controlled state, feed, and batch parameters

**pH.** Unlike Ding (a single starting pH), pH is *held* at 5 / 7 / 9 here, so each condition gets
its own fixed `S_H_ion = 10^(-pH)`, carried as a constant. This is the lever that should make
e-ADM suppress methanogens at pH 5 and 9 and permit methane at pH 7 — matching the paper.

**Feed.** Reactors ran at **~15 gCOD L⁻¹** food waste (paper methods). The detailed carbohydrate/
lipid/protein split lives in the paper's Table S1 (not in the public repo); until that is wired in
we reuse the food-waste characterisation from the Ding notebook as a documented placeholder.

**Batch reactor.** Closed batch, 250 mL working volume. We keep the same base-parameter operating
point as the calibrated model (`q_in=0`, small `V_liq`/`V_gas`, `k_p=0`) so the held-out test runs
at the identical configuration the model was trained on.


In [ ]:
p = utils.load_model_json(str(MODEL_DB), "e_adm")
Ka = p["model_parameters"]

feed = core.Feed(name="Foodwaste", carbohydrates=50.1, lipids=21.5, proteins=20.5,
                 tss=30, si=47, xi=22.5)                            # PLACEHOLDER: paper Table S1 pending
TOTAL_FEED_COD = 15.0                                               # gCOD/L (paper methods)
TSS = TOTAL_FEED_COD * feed.tss / 100
TDS = TOTAL_FEED_COD - TSS

def s_h(ph):
    return 10.0 ** (-ph)

print(f"feed: {feed.name}, total COD = {TOTAL_FEED_COD} gCOD/L  (TSS={TSS:.1f}, TDS={TDS:.1f})")
print("pH -> S_H_ion:", {ph: f"{s_h(ph):.1e}" for ph in PHS})

## 6. The microbial seed — from the **provided** rep-seqs + feature table (HMMER markers)

Day-0 biomass initial conditions come from the 16S community, mapped to e-ADM functional groups by
the marker pipeline, starting from the repo's **already-denoised** `16S_rep_seq.fasta` +
`16S_feature_table.txt` (no amplicon rerun). Only the D0 samples of the slice (J1-J9) are needed.

The recipe (run once; needs the GTDB amplicon-to-genome reference — `adtoolbox database
download-all-databases`):

1. **VSEARCH** the D0 rep-seqs against GTDB SSU at **0.95** identity -> representative genome per
   ASV. (0.95 rather than 0.97 lifts abundance-weighted coverage to **~99.5 %** of each sample.)
2. **Download** the matched representative genomes (NCBI Datasets / HTTPS).
3. **`annotate_genome_with_marker_hmms`** per genome: Prodigal -> `hmmsearch` against the marker
   **profile HMMs** with the **adaptive per-marker cutoffs** (`Marker_Profile_Cutoffs.csv`).
   *We use the **HMMER** backend, not MMseqs: the calibrated profile cutoffs are what make the
   presence calls reliable; MMseqs uses generic score thresholds.*
4. **`aggregate_genome_cod`**: weight each genome's group-COD by its ASV abundance in the sample
   (`genome_marker_evidence` auto-routes the `.domtbl` through the HMMER cutoffs).

The resulting `cod_profile.csv` files are **provided** under `tutorial_output/rico_day0/<J-code>/`,
so the reader below fills biomass without re-running the pipeline. Because inoculum was
**DNA-normalised** (equal microbial loading across treatments), total day-0 biomass COD is ~constant
across the slice, so composition (the COD shares) is what differs — scaled by one nominal
`INOCULUM_COD`. (This community is acidogenic: methanogens `X_Me_ac`/`X_Me_CO2` are ~0, unlike Ding.)

In [ ]:
# --- D0 samples of the slice: metadata J-code <-> condition ---
meta = pd.read_csv(RAW / "metadata_interactions.txt", sep="\t", dtype=str)
d0 = meta[(meta.type == "experiment") & (meta.feedstock == FEEDSTOCK) &
          (meta.inoculum == INOCULUM) & (meta.temperature == str(TEMPERATURE)) &
          (meta.pH.isin([str(x) for x in PHS])) & (meta.day == "D0")]
d0_samples = {f"pH{r.pH}_rep{r.rep}": r.SampleID for _, r in d0.iterrows()}
print("day-0 16S samples (condition -> J-code):")
for k, v in sorted(d0_samples.items()):
    print(f"  {k:12} {v}")

INOCULUM_COD = 1.0        # gCOD/L nominal day-0 biomass per reactor (DNA-normalised => ~constant)

# The HMMER marker pipeline (steps 1-4 in the markdown) has already been run; its per-sample
# cod_profile.csv files are provided under SEED_OUT. Flip RUN_SEED=True to regenerate them from
# scratch (needs the GTDB amplicon-to-genome DB) via:
#   vsearch --usearch_global 16S_rep_seq.fasta --db <GTDB ssu_all.fna> --id 0.95 ...   (ASV->genome)
#   download matched genomes  ->  mg.annotate_genome_with_marker_hmms(...)  (Prodigal + hmmsearch)
#   mg.aggregate_genome_cod(genome_cods, genome_abundances, normalize=True) -> cod_profile.csv
RUN_SEED = False
if RUN_SEED:
    raise NotImplementedError(
        "Regenerate SEED_OUT/<J-code>/cod_profile.csv with the HMMER marker pipeline "
        "(see the recipe above), then set RUN_SEED=False and re-run.")
else:
    n = sum((SEED_OUT / j / "cod_profile.csv").exists() for j in d0_samples.values())
    print(f"\nusing provided HMMER cod_profiles: {n}/{len(d0_samples)} present in {SEED_OUT}")

In [ ]:
def biomass_ic(condition):
    """Day-0 X_* biomass (gCOD/L) = functional-group COD shares x INOCULUM_COD.

    Reads SEED_OUT/<J-code>/cod_profile.csv produced by the marker pipeline. Returns {} (=> model
    defaults) when the seed step has not been run yet, so the object still assembles and saves.
    """
    jcode = d0_samples.get(condition)
    prof_path = SEED_OUT / str(jcode) / "cod_profile.csv"
    if not (jcode and prof_path.exists()):
        return {}
    prof = pd.read_csv(prof_path).set_index("group")["value"]
    prof = prof / prof.sum()
    return {g: float(prof[g]) * INOCULUM_COD for g in prof.index}

_probe = biomass_ic("pH5_repA")
print("biomass seed present?", bool(_probe),
      "-- HMMER cod_profiles found" if _probe else "-- run the gated seed step first")

## 7. Assemble training/validation-ready `Experiment` objects

Same construction as the Ding notebook: VFAs at t₀ become initial conditions, the acid–base pairs
are split by the fixed pH, biomass is seeded from the metagenomics COD shares (when available), and
the batch operating point is attached. Here every condition also pins its own `S_H_ion` constant.


In [ ]:
def build_experiment(ph, rep):
    times, data = trajectory(ph, rep)                              # (time,), (time, VFA) gCOD/L
    S_H = s_h(ph)
    ic = {v: float(data[0, j]) for j, v in enumerate(VARIABLES)}   # VFAs at t0
    ic.update(biomass_ic(f"pH{ph}_rep{rep}"))                      # microbial seed (if seeded)
    for v, ka in [("S_va", "K_a_va"), ("S_bu", "K_a_bu"), ("S_pro", "K_a_pro"),
                  ("S_cap", "K_a_cap"), ("S_ac", "K_a_ac")]:
        ic[f"{v}_ion"] = Ka[ka] / (Ka[ka] + S_H) * ic[v]
    ic.update({"S_su": 0.0, "S_aa": 0.0, "S_fa": 0.0,
               "TSS": TSS, "TDS": TDS, "S_H_ion": S_H})
    return core.Experiment(
        name=f"FW_A_35C_pH{ph}_rep{rep}",
        time=[float(t) for t in times], variables=list(VARIABLES),
        data=data.T.tolist(),                                      # core.Experiment wants (variable, time)
        feed=feed, initial_concentrations=ic,
        base_parameters={"q_in": 0, "V_liq": 0.0001, "V_gas": 0.00007},
        constants=["S_H_ion"], reference=DOI, model_name="e_adm")

experiments = [build_experiment(ph, rep) for ph in PHS for rep in REPS]
e = experiments[0]
print(f"built {len(experiments)} experiments")
print(f"{e.name}: {len(e.time)} timepoints, variables={e.variables}")
print("  t0 VFAs:", {v: round(e.initial_concentrations[v], 3) for v in VARIABLES})
print("  S_H_ion:", e.initial_concentrations["S_H_ion"])

## 8. Sanity checks

Confirm the objects round-trip through `core.Experiment` (which transposes data on load), that the
pH → acid spectrum trend is present, and that the shape matches the Ding training objects.


In [ ]:
# (a) round-trip: data stored as (variable, time), Experiment transposes back to (time, variable)
e = experiments[0]
assert np.asarray(e.data, float).shape == (len(e.time), len(e.variables))

# (b) pH trend: butyrate+caproate (chain elongation) should peak at pH 5, acetate share rise at pH 9
print("final-day VFA spectrum by pH (rep A, gCOD/L):")
for ph in PHS:
    _, d = trajectory(ph, "A")
    last = dict(zip(VARIABLES, d[-1]))
    ce = last["S_bu"] + last["S_cap"]
    print(f"  pH {ph}:  S_ac={last['S_ac']:.2f}  S_bu={last['S_bu']:.2f}  "
          f"S_cap={last['S_cap']:.2f}   (C4+C6 = {ce:.2f})")

# (c) shape matches Ding experiments
ding = REPO / "Examples" / "Studies" / "ding_experiments.json"
if ding.exists():
    dj = json.load(open(ding)); k0 = next(iter(dj))
    print("\nDing experiment keys :", sorted(dj[k0].keys()))
    print("this matches build_experiment output fields (name/time/variables/data/feed/ic/...).")

## 9. Save — ready for parameter tuning / validation

Written in the same JSON layout as `ding_experiments.json` so the tuning and modeling notebooks can
load it unchanged. Load the calibrated model, run each condition, and compare predicted vs. observed
VFA trajectories — a genuine held-out test of the acidogenesis + chain-elongation + pH machinery.


In [ ]:
out = REPO / "Examples" / "Studies" / "rico_experiments.json"
payload = {}
for e in experiments:
    payload[e.name] = {
        "name": e.name,
        "condition": e.name.rsplit("_rep", 1)[0],
        "reference": e.reference, "sra_bioproject": SRA, "model_name": e.model_name,
        "time": e.time, "variables": e.variables,
        "data": np.asarray(e.data, float).T.tolist(),             # (variable, time)
        "feed": {"name": feed.name, "carbohydrates": feed.carbohydrates, "lipids": feed.lipids,
                 "proteins": feed.proteins, "tss": feed.tss, "si": feed.si, "xi": feed.xi},
        "initial_concentrations": e.initial_concentrations,
        "base_parameters": e.base_parameters, "constants": e.constants,
    }
out.write_text(json.dumps(payload, indent=1))
print("wrote", out, "(", len(payload), "experiments )")
print("biomass seeded:", any(any(k.startswith("X_") for k in v["initial_concentrations"])
                             for v in payload.values()), "(HMMER marker profiles)")

## 10. Ordination — do the day-0 communities separate by pH?

A quick sanity check on the seed itself: ordinate the 9 day-0 functional-COD profiles
(Bray–Curtis) in **3-D NMDS**, coloured by pH, with a PERMANOVA test. The inocula were acclimated
to their operating pH for ~2 months, so we expect pH structure even at t₀ — and indeed pH explains
most of the between-sample variance, with the two *inhibitory* extremes (pH 5 and 9) converging on a
similar acidogenic community while **pH 7 (methanogenesis-permissive) stands apart**.

In [ ]:
import plotly.graph_objects as go
from scipy.spatial.distance import pdist, squareform
from sklearn.manifold import MDS

# 9 day-0 COD profiles -> samples x groups matrix
prof = {c: pd.read_csv(SEED_OUT / j / "cod_profile.csv").set_index("group")["value"]
        for c, j in d0_samples.items()}
wide = pd.DataFrame(prof).T.fillna(0.0)
labels = list(wide.index)
ph = [l.split("_")[0] for l in labels]                          # pH5 / pH7 / pH9
dist = squareform(pdist(wide.to_numpy(float), metric="braycurtis"))

def permanova(dist, g, perm=999, seed=0):
    g = np.asarray(g); n = len(g); d2 = dist ** 2
    lev = np.unique(g); a = len(lev); sst = d2[np.triu_indices(n, 1)].sum() / n
    def ssw(l):
        s = 0.0
        for L in lev:
            idx = np.where(l == L)[0]; k = len(idx)
            if k > 1: s += d2[np.ix_(idx, idx)][np.triu_indices(k, 1)].sum() / k
        return s
    w = ssw(g); F = ((sst - w) / (a - 1)) / (w / (n - a)); r2 = (sst - w) / sst
    rng = np.random.default_rng(seed); ge = 1
    for _ in range(perm):
        wp = ssw(rng.permutation(g)); fp = ((sst - wp) / (a - 1)) / (wp / (n - a)) if wp > 0 else np.inf
        if fp >= F: ge += 1
    return F, r2, ge / (perm + 1)

F, R2, p = permanova(dist, ph)
nmds = MDS(n_components=3, metric_mds=False, dissimilarity="precomputed",
           random_state=0, n_init=16, max_iter=800, normalized_stress=True)
xyz = nmds.fit_transform(dist); stress = nmds.stress_
print(f"3-D NMDS stress={stress:.3f}  |  PERMANOVA by pH: F={F:.1f}, R2={R2:.2f}, p={p:.3f}")

COL = {"pH5": "#d55e00", "pH7": "#009e73", "pH9": "#0072b2"}
fig = go.Figure()
for g in ["pH5", "pH7", "pH9"]:
    idx = [i for i, gg in enumerate(ph) if gg == g]
    fig.add_trace(go.Scatter3d(
        x=xyz[idx, 0], y=xyz[idx, 1], z=xyz[idx, 2], mode="markers", name=g,
        text=[labels[i] for i in idx], hovertemplate="%{text}<extra>" + g + "</extra>",
        marker=dict(size=7, color=COL[g], line=dict(width=1, color="white"))))
fig.update_layout(
    template="plotly_white", height=560,
    title=dict(text=f"Day-0 community COD profiles &middot; 3-D NMDS (Bray-Curtis)<br>"
               f"<sup>stress={stress:.3f} &middot; PERMANOVA by pH: F={F:.1f}, R\u00b2={R2:.2f}, "
               f"{'p&lt;0.001' if p < 0.001 else f'p={p:.3f}'}</sup>", x=0.5),
    scene=dict(xaxis_title="NMDS1", yaxis_title="NMDS2", zaxis_title="NMDS3"),
    legend=dict(title="pH"))
fig.show()

## What you built

A **held-out validation set** — 9 batch food-waste reactors (pH 5/7/9 × 3 reps, 35 °C, sludge
inoculum) with C2–C6 VFA trajectories in gCOD L⁻¹, each pinned at its controlled pH — assembled
entirely from Rico et al.'s **provided** rep-seqs, feature table, and FA tables (no amplicon rerun).

**Complete now:** VFA targets, feed, per-condition pH, batch parameters, and the full object saved
to `Studies/rico_experiments.json`.

**One remaining step for biomass initial conditions:** download the GTDB amplicon-to-genome
reference (`adtoolbox database download-all-databases`), run the marker pipeline on the provided
rep-seqs + feature table for the D0 samples, then flip `RUN_SEED` — the reader in §6 will fill the
`X_*` seed automatically.

**How to use it:** load `reference_data/calibrated_model.json` in the modeling/tuning notebook, run
each condition, and compare to these trajectories. Because this is **arrested AD**, judge the model
on the **VFA spectrum vs. pH** (and the on/off pattern of methanogenesis), not on methane yield.
